In [1]:
#train.csv로 모델을 학습, 학습한 모델을 test.csv에 적용
#bank marketing 참여 여부(정기 예금 가입 여부)를 예측하는 classification tast.

In [2]:
#특성 엔지니어링 (변수 손질)

In [3]:
#!pip install catboost

In [4]:
import numpy as np
import os
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgbm
from catboost import CatBoostClassifier
from sklearn.ensemble import BaggingClassifier, VotingClassifier

import warnings
warnings.filterwarnings('ignore')

In [5]:
df_trn = pd.read_csv('train.csv')
df_tst=pd.read_csv('test.csv')

In [7]:
df_trn

,ID,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,label
0,train00001,34,blue-collar,married,primary,no,358,yes,no,unknown,23,may,100,4,-1,0,unknown,0
1,train00002,33,blue-collar,married,secondary,no,-53,yes,no,unknown,20,may,172,1,-1,0,unknown,0
2,train00003,32,management,single,tertiary,no,207,yes,no,cellular,2,feb,56,1,-1,0,unknown,0
3,train00004,37,blue-collar,divorced,secondary,no,638,yes,no,cellular,8,may,326,1,326,2,success,0
4,train00005,33,housemaid,married,secondary,no,826,yes,no,cellular,11,may,256,1,-1,0,unknown,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31642,train31643,60,retired,married,secondary,no,3932,yes,no,unknown,14,may,407,2,-1,0,unknown,0
31643,train31644,44,blue-collar,married,unknown,no,15578,yes,no,unknown,29,may,401,1,-1,0,unknown,0
31644,train31645,54,unemployed,single,secondary,no,3611,yes,no,cellular,5,feb,81,1,-1,0,unknown,0
31645,train31646,33,services,married,secondary,yes,2,no,no,cellular,6,may,136,5,-1,0,unknown,0


In [8]:
print((df_trn['label'] == 1).mean()*100)

11.697791259835055


In [9]:
month_map = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}

df_trn['month'] = df_trn['month'].map(month_map)
df_tst['month'] = df_tst['month'].map(month_map)

In [10]:
df_trn.head()

,ID,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,label
0,train00001,34,blue-collar,married,primary,no,358,yes,no,unknown,23,5,100,4,-1,0,unknown,0
1,train00002,33,blue-collar,married,secondary,no,-53,yes,no,unknown,20,5,172,1,-1,0,unknown,0
2,train00003,32,management,single,tertiary,no,207,yes,no,cellular,2,2,56,1,-1,0,unknown,0
3,train00004,37,blue-collar,divorced,secondary,no,638,yes,no,cellular,8,5,326,1,326,2,success,0
4,train00005,33,housemaid,married,secondary,no,826,yes,no,cellular,11,5,256,1,-1,0,unknown,0


In [11]:
y_cols=['label']
num_cols=['age', 'balance', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous']
cat_cols=['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'poutcome']
id_cols=['ID']
x_cols=num_cols+cat_cols

In [12]:
df_trn_x = df_trn[x_cols]
df_tst_x = df_tst[x_cols]

df_trn_y = df_trn[y_cols]
df_tst_y = df_tst[y_cols]

In [13]:
# 1. 연령대(age) 범주화
df_trn_x['age_group'] = pd.cut(df_trn_x['age'], bins=[0, 30, 60, 100], labels=['young', 'middle_aged', 'senior'])
df_tst_x['age_group'] = pd.cut(df_tst_x['age'], bins=[0, 30, 60, 100], labels=['young', 'middle_aged', 'senior'])

# 2. 잔고(balance) 범주화
df_trn_x['balance_group'] = pd.cut(df_trn_x['balance'], bins=[-1000, 0, 5000, 10000, 50000], labels=['negative', 'low', 'medium', 'high'])
df_tst_x['balance_group'] = pd.cut(df_tst_x['balance'], bins=[-1000, 0, 5000, 10000, 50000], labels=['negative', 'low', 'medium', 'high'])

# 3. 캠페인 참여 기간(duration * campaign)
df_trn_x['campaign_duration'] = df_trn_x['duration'] * df_trn_x['campaign']
df_tst_x['campaign_duration'] = df_tst_x['duration'] * df_tst_x['campaign']

In [14]:
for col in cat_cols:
    df_trn_x[col] = df_trn_x[col].astype('category')
    df_tst_x[col] = df_tst_x[col].astype('category')

In [15]:
y_trn = df_trn[y_cols]

# 1. 수치형 변수와 타겟 변수 간의 피어슨 상관계수 계산
num_data = df_trn[num_cols]  # 훈련 데이터에서 수치형 변수만 선택
correlation = num_data.corrwith(y_trn.squeeze())  # y_trn.squeeze()는 y를 1D로 변환

# 결과 출력
print("수치형 변수와 타겟 변수 간의 피어슨 상관계수:")
print(correlation)


수치형 변수와 타겟 변수 간의 피어슨 상관계수:
age         0.034886
balance     0.051078
day        -0.026003
month       0.022448
duration    0.391085
campaign   -0.074137
pdays       0.097961
previous    0.086401
dtype: float64


In [16]:
import pandas as pd
from scipy.stats import chi2_contingency
cat_data = df_trn[cat_cols]  # 훈련 데이터에서 범주형 변수만 선택
cat_results = {}

for cat_col in cat_data.columns:
    # 타겟 변수와 범주형 변수 간의 교차표 생성
    contingency_table = pd.crosstab(cat_data[cat_col], y_trn.squeeze())
    
    # 카이제곱 검정 수행
    chi2_stat, p_val, dof, expected = chi2_contingency(contingency_table)
    
    # p-value가 0.05보다 작으면 유의미한 관계가 있다고 판단
    cat_results[cat_col] = {
        'Chi2 Stat': chi2_stat,
        'p-value': p_val
    }

# 결과 출력
print("\n범주형 변수와 타겟 변수 간의 카이제곱 검정 결과:")
for cat_col, result in cat_results.items():
    print(f"{cat_col}: Chi2 Stat = {result['Chi2 Stat']}, p-value = {result['p-value']}")


범주형 변수와 타겟 변수 간의 카이제곱 검정 결과:
job: Chi2 Stat = 633.3990759175488, p-value = 9.985967839819186e-129
marital: Chi2 Stat = 105.98217959771111, p-value = 9.688624186468241e-24
education: Chi2 Stat = 151.80084659109377, p-value = 1.0771479229578099e-32
default: Chi2 Stat = 10.839195283810286, p-value = 0.0009937391014869608
housing: Chi2 Stat = 644.9101538856866, p-value = 2.8579790984445928e-142
loan: Chi2 Stat = 141.76727087128427, p-value = 1.0933750110012598e-32
contact: Chi2 Stat = 705.7146270366479, p-value = 5.7018506795105544e-154
poutcome: Chi2 Stat = 3051.947321355698, p-value = 0.0


In [17]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# 범주형 변수+특성 엔지니어링한 변수를 Label Encoding으로 변환
cat_cols_with_new = cat_cols + ['age_group', 'balance_group'] 
label_encoders = {}  # 각 컬럼에 대한 LabelEncoder 객체 저장

for col in cat_cols_with_new:
    le = LabelEncoder()
    df_trn_x[col] = le.fit_transform(df_trn_x[col])  # 훈련 데이터 변환
    df_tst_x[col] = le.transform(df_tst_x[col])      # 테스트 데이터 변환
    label_encoders[col] = le

In [18]:
n_positive = sum(df_trn_y.iloc[:, 0] == 1)  # 양성 클래스의 샘플 수
n_negative = sum(df_trn_y.iloc[:, 0] == 0)  # 음성 클래스의 샘플 수
# scale_pos_weight 설정 (양성/음성 비율)
scale_pos_weight = n_negative / n_positive

In [19]:
# 최종 LightGBM: 최초 RandomizedSearchCV(7-fold)에서 선정된 learning_rate=0.11 사용
# (탐색 시 depth, iterations, l2_leaf_reg, subsample은 CatBoost 파라미터명이라 LightGBM에 적용되지 않아 제외)
best_model = lgbm.LGBMClassifier(learning_rate=0.11,
                                 scale_pos_weight=scale_pos_weight,
                                 verbose=-1)

In [20]:
bagging_model = BaggingClassifier(estimator=best_model,
                                 n_estimators=50,  # 모델 개수
                                 random_state=42)

In [21]:
catboost_model = CatBoostClassifier(iterations=1000, 
                                    depth=10, 
                                    learning_rate=0.1, 
                                    scale_pos_weight=scale_pos_weight,
                                    loss_function='Logloss', 
                                    verbose=False)

In [22]:
bagging_catboost_model = BaggingClassifier(estimator=catboost_model,
                                          n_estimators=50, 
                                          random_state=42)

In [23]:
ensemble_model = VotingClassifier(estimators=[
    ('lightgbm_bagging', bagging_model),  # Bagging + LightGBM 모델
    ('catboost_bagging', bagging_catboost_model)  # Bagging + CatBoost 모델
], voting='soft') 

In [24]:
# 훈련 데이터를 훈련셋과 검증셋으로 분리
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(df_trn_x, df_trn_y, test_size=0.2, random_state=42)

In [25]:
# 앙상블 모델 학습
ensemble_model.fit(X_train, y_train)

  File "C:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 199, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\ProgramData\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


VotingClassifier(estimators=[('lightgbm_bagging',
                              BaggingClassifier(estimator=LGBMClassifier(learning_rate=0.11,
                                                                         scale_pos_weight=7.548622366288493,
                                                                         verbose=-1),
                                                n_estimators=50,
                                                random_state=42)),
                             ('catboost_bagging',
                              BaggingClassifier(estimator=<catboost.core.CatBoostClassifier object at 0x000001F0E1897590>,
                                                n_estimators=50,
                                                random_state=42))],
                 voting='soft')

In [26]:
# 검증 데이터에 대해 예측
y_val_pred = ensemble_model.predict(X_val)

In [27]:
# 검증 데이터 성능 평가 (F1 score 사용)
f1 = f1_score(y_val, y_val_pred)
print(f"검증 데이터 F1 Score: {f1}")

검증 데이터 F1 Score: 0.6180555555555556
